In [1]:
import os
import sqlite3
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


 Opdracht 5

 Opdracht 5.a



 Reset, Create en Load Database functies

In [ ]:


def reset_sdm():
    #Verwijdert alle bestaande tabellen in de SDM database.
    with sqlite3.connect('sdm.db') as conn:
        tables = pd.read_sql(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';", conn)
        cursor = conn.cursor()
        for table in tables['name']:
            cursor.execute(f"DROP TABLE IF EXISTS {table}")
        conn.commit()
    print("Database succesvol geleegd.")


def create_sdm_tables():
    #Leest het SQL bestand in en bouwt de tabellen op in de SDM database.
    sql_file = 'BikeToDrive_RIM - alle bestanden.txt'
    if not os.path.exists(sql_file):
        print(f"Let op: {sql_file} niet gevonden!")
        return

    with open(sql_file, 'r', encoding='utf-8') as f:
        sql = f.read()

    with sqlite3.connect('sdm.db') as sdm_conn:
        cursor = sdm_conn.cursor()
        # Splits de SQL statements en negeer lege regels of comments
        statements = [stmt.strip() for stmt in sql.split(
            ';') if stmt.strip() and not stmt.strip().startswith('--')]

        for stmt in statements:
            try:
                cursor.execute(stmt)
            except Exception as e:
                print(f"Error bij uitvoeren DDL: {e} voor {stmt[:50]}...")
        sdm_conn.commit()
    print("Tabellen succesvol aangemaakt in SDM.")


def load_data_to_sdm():
    #Laadt data van de bron-databases naar het SDM met behulp van mappings.
    mappings = {
        'BikeToDrive_1_Accessoireverkoop.db': {
            'Accessoire_Verkoop': 'Accessoireverkoop',
            'Monteur': 'Accessoireverkoop_Monteur',
            'Leverancier': 'Accessoireverkoop_Leverancier',
            'Accessoire': 'Accessoireverkoop_Accessoire',
            'Filiaal': 'Accessoireverkoop_filiaal',
            'Klant': 'Accessoireverkoop_klant'
        },
        'BikeToDrive_2_Fietsverkoop.db': {
            'Fiets_Verkoop': 'Fietsverkoop',
            'Fiets': 'Fietsverkoop_Fiets',
            'Monteur': 'Fietsverkoop_Monteur',
            'Fabrikant': 'Fietsverkoop_Fabrikant',
            'Filiaal': 'Fietsverkoop_filiaal',
            'Klant': 'Fietsverkoop_klant'
        },
        'BikeToDrive_3_Onderhoud.db': {
            'Onderhoud': 'Onderhoud',
            'Fiets': 'Onderhoud_Fiets',
            'Monteur': 'Onderhoud_Monteur',
            'Fabrikant': 'Onderhoud_Fabrikant',
            'Filiaal': 'Onderhoud_filiaal'
        },
        'BikeToDrive_4_Accessoire_Inkoop.db': {
            'Accessoire_Inkoop': 'Accessoire_inkoop',
            'Accessoire': 'Accessoire_inkoop_Accessoire',
            'Leverancier': 'Accessoire_inkoop_Leverancier'
        },
        'BikeToDrive_5_Fiets_Inkoop.db': {
            'Fiets_Inkoop': 'Fiets_Inkoop',
            'Fiets': 'Fiets_Inkoop_Fiets',
            'Fabrikant': 'Fiets_Inkoop_Fabrikant'
        }
    }

    with sqlite3.connect('sdm.db') as sdm_conn:
        for db, table_map in mappings.items():
            if not os.path.exists(db):
                print(f'Bron DB {db} niet gevonden. Wordt overgeslagen.')
                continue

            with sqlite3.connect(db) as conn:
                for src_table, dst_table in table_map.items():
                    try:
                        df = pd.read_sql(f"SELECT * FROM {src_table}", conn)
                        df.to_sql(dst_table, sdm_conn,
                                  if_exists='append', index=False)
                        print(
                            f'Geladen: {len(df)} rijen van {src_table} -> {dst_table}')
                    except Exception as e:
                        print(f'Fout bij inladen {src_table}: {e}')

    print("Data laden completed.")


def full_refresh_sdm():
    # Voert de volledige ETL pijplijn uit: Reset -> Create DDL -> Load Data.
    print("Start Full Refresh...")
    reset_sdm()
    create_sdm_tables()
    load_data_to_sdm()
    print("Full refresh van SDM completed.")


 Opdracht 5.b

 Database testen en exploreren

In [3]:
dbs = [
    'BikeToDrive_1_Accessoireverkoop.db',
    'BikeToDrive_2_Fietsverkoop.db',
    'BikeToDrive_3_Onderhoud.db',
    'BikeToDrive_4_Accessoire_Inkoop.db',
    'BikeToDrive_5_Fiets_Inkoop.db'
]

# Even checken of de tabellen kloppen
for db in dbs:
    if os.path.exists(db):
        with sqlite3.connect(db) as conn:
            tables = pd.read_sql(
                "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'", conn)
            print(f'{db}: {list(tables["name"])}')
    else:
        print(f'{db} not found')


BikeToDrive_1_Accessoireverkoop.db: ['Accessoire_Verkoop', 'Monteur', 'Leverancier', 'Accessoire', 'Filiaal', 'Klant']
BikeToDrive_2_Fietsverkoop.db: ['Fiets_Verkoop', 'Fiets', 'Monteur', 'Fabrikant', 'Filiaal', 'Klant']
BikeToDrive_3_Onderhoud.db: ['Onderhoud', 'Fiets', 'Monteur', 'Fabrikant', 'Filiaal']
BikeToDrive_4_Accessoire_Inkoop.db: ['Accessoire_Inkoop', 'Accessoire', 'Leverancier']
BikeToDrive_5_Fiets_Inkoop.db: ['Fiets_Inkoop', 'Fiets', 'Fabrikant']


In [4]:
shared_tables = ['Klant', 'Monteur', 'Filiaal',
                 'Leverancier', 'Accessoire', 'Fiets', 'Fabrikant']

print("\n--- Aantal unieke ID's in gedeelde tabellen over de databases ---")
for table in shared_tables:
    ids = set()
    for db in dbs:
        if not os.path.exists(db):
            continue
        try:
            with sqlite3.connect(db) as conn:
                # Soms heet de Primary Key kolom in schoolopdrachten 'id' en soms de tabelnaam+'nr' (bijv. klantnr)
                # Als 'id' foutmeldingen geeft, probeer het dan aan te passen.
                df = pd.read_sql(f"SELECT * FROM {table}", conn)

                # Bepaal de naam van de eerste kolom (vaak de PK) en sla die op in de set
                pk_col = df.columns[0]
                ids.update(df[pk_col].tolist())
        except:
            pass
    print(f'{table}: {len(ids)} unique IDs gevonden')



--- Aantal unieke ID's in gedeelde tabellen over de databases ---
Klant: 26 unique IDs gevonden
Monteur: 15 unique IDs gevonden
Filiaal: 5 unique IDs gevonden
Leverancier: 5 unique IDs gevonden
Accessoire: 13 unique IDs gevonden
Fiets: 75 unique IDs gevonden
Fabrikant: 11 unique IDs gevonden


In [5]:
# Kijken hoe 'Klant' eruit ziet in Accessoireverkoop
if os.path.exists('BikeToDrive_1_Accessoireverkoop.db'):
    with sqlite3.connect('BikeToDrive_1_Accessoireverkoop.db') as conn:
        cursor = conn.cursor()
        cursor.execute("PRAGMA table_info(Klant)")
        columns = cursor.fetchall()
        print("Klant columns in Accessoireverkoop:",
              [col[1] for col in columns])
        df = pd.read_sql("SELECT * FROM Klant LIMIT 5", conn)
        print(df.head())


Klant columns in Accessoireverkoop: ['klantnr', 'naam', 'adres', 'woonplaats', 'geslacht', 'geboortedatum']
   klantnr           naam           adres               woonplaats geslacht  \
0        1     Jan Jansen   Kerkstraat 12  Gewijzigde Test Locatie        M   
1        3  Pieter Visser   Havenstraat 3                 Den Haag        M   
2        4      Emma Smit    Boomgaard 22                  Haarlem        V   
3        5     Tom Bakker  Stationsweg 44                   Leiden        M   
4        6    Lisa Meijer   Dijkstraat 10                  Zaandam        V   

  geboortedatum  
0    1985-03-22  
1    1978-11-05  
2    1995-02-18  
3    1982-09-09  
4    1993-12-30  


 Opdracht 6 - Mutaties en CDC Testing

In [6]:
bron_db = 'BikeToDrive_1_Accessoireverkoop.db'

# LET OP: Check of de primary key 'klantnr' heet. Mocht dit 'id' zijn,
# verander 'klantnr' hieronder dan overal naar 'id'.
PK_KOLOM = 'klantnr'

if os.path.exists(bron_db):
    with sqlite3.connect(bron_db) as conn_bron:
        cursor_bron = conn_bron.cursor()

        try:
            # 1. INSERT
            cursor_bron.execute(
                f"INSERT INTO Klant ({PK_KOLOM}, naam, woonplaats) VALUES (9999, 'Test Klant Opdracht 6', 'SDM-Stad')")

            # 2. UPDATE
            cursor_bron.execute(
                f"UPDATE Klant SET woonplaats = 'Gewijzigde Test Locatie' WHERE {PK_KOLOM} = 1")

            # 3. DELETE
            cursor_bron.execute(f"DELETE FROM Klant WHERE {PK_KOLOM} = 2")

            conn_bron.commit()
            print(
                "\nStap 1: INSERT, UPDATE en DELETE succesvol uitgevoerd op de bron-database.")

        except Exception as e:
            print(f"Er ging iets mis in de brondatabase. Foutmelding: {e}")
            # Mocht het falen wegens de PK naam, rolt dit netjes terug
            conn_bron.rollback()
else:
    print(f"Brondatabase {bron_db} niet gevonden voor Opdracht 6.")


print("\nStarten met Full Refresh van het SDM om wijzigingen op te halen...")
# Omdat full_refresh_sdm nu alles doet, hoeven we load_data_to_sdm() niet nogmaals aan te roepen!
full_refresh_sdm()
print("Stap 2: SDM succesvol opnieuw opgebouwd en geladen met de nieuwste data.")


print("\n--- Test Resultaten in SDM (Tabel: Accessoireverkoop_klant) ---")
with sqlite3.connect('sdm.db') as sdm_connection:
    try:
        df_insert = pd.read_sql(
            f"SELECT * FROM Accessoireverkoop_klant WHERE {PK_KOLOM} = 9999", sdm_connection)
        print(f"\n1. INSERT test (Is de nieuwe klant 9999 aanwezig?):")
        if not df_insert.empty:
            print("✅ Succes! Klant gevonden:")
            print(df_insert)
        else:
            print("❌ Gefaald. Klant niet gevonden.")

        df_update = pd.read_sql(
            f"SELECT * FROM Accessoireverkoop_klant WHERE {PK_KOLOM} = 1", sdm_connection)
        print(f"\n2. UPDATE test (Heeft klant 1 de nieuwe woonplaats?):")
        if not df_update.empty:
            print(
                "✅ Controleer de onderstaande data op de wijziging (Gewijzigde Test Locatie):")
            print(df_update)
        else:
            print("❌ Gefaald. Klant 1 bestaat niet in SDM.")

        df_delete = pd.read_sql(
            f"SELECT * FROM Accessoireverkoop_klant WHERE {PK_KOLOM} = 2", sdm_connection)
        print(f"\n3. DELETE test (Is klant 2 succesvol verwijderd uit SDM?):")
        if df_delete.empty:
            print("✅ Succes! Klant 2 bestaat niet meer in het SDM.")
        else:
            print("❌ Gefaald. Klant 2 is nog steeds aanwezig.")

    except Exception as e:
        print(f"Er is een fout opgetreden bij het valideren van het SDM: {e}")



Stap 1: INSERT, UPDATE en DELETE succesvol uitgevoerd op de bron-database.

Starten met Full Refresh van het SDM om wijzigingen op te halen...
Start Full Refresh...
Database succesvol geleegd.
Tabellen succesvol aangemaakt in SDM.
Geladen: 100 rijen van Accessoire_Verkoop -> Accessoireverkoop
Geladen: 10 rijen van Monteur -> Accessoireverkoop_Monteur
Geladen: 5 rijen van Leverancier -> Accessoireverkoop_Leverancier
Geladen: 10 rijen van Accessoire -> Accessoireverkoop_Accessoire
Geladen: 4 rijen van Filiaal -> Accessoireverkoop_filiaal
Fout bij inladen Klant: Execution failed
Geladen: 150 rijen van Fiets_Verkoop -> Fietsverkoop
Geladen: 75 rijen van Fiets -> Fietsverkoop_Fiets
Geladen: 10 rijen van Monteur -> Fietsverkoop_Monteur
Geladen: 10 rijen van Fabrikant -> Fietsverkoop_Fabrikant
Geladen: 4 rijen van Filiaal -> Fietsverkoop_filiaal
Geladen: 25 rijen van Klant -> Fietsverkoop_klant
Geladen: 50 rijen van Onderhoud -> Onderhoud
Geladen: 30 rijen van Fiets -> Onderhoud_Fiets
Gelade